In [1]:
from dotenv import load_dotenv
from IPython.display import display, Markdown
# AutoGen's wrapper:

from autogen_ext.tools.langchain import LangChainToolAdapter
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken

# LangChain tools:

from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain.agents import Tool

In [2]:
load_dotenv(override=True)

True

In [3]:
prompt = """Your task is to find a one-way flight from IAH to BLR in June 2025.
First search online for promising deals.
Next, write all the deals to a file called flights.md with full details.
Finally, select the one you think is cheepest price and reply with a short summary.
Reply with the selected flight only, and only after you have written the details to the file."""


serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)
autogen_tools = [autogen_serper]

langchain_file_management_tools = FileManagementToolkit(root_dir="sandbox").get_tools()
for tool in langchain_file_management_tools:
    autogen_tools.append(LangChainToolAdapter(tool))

for tool in autogen_tools:
    print(tool.name, tool.description)

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
agent = AssistantAgent(name="searcher", model_client=model_client, tools=autogen_tools, reflect_on_tool_use=True)
message = TextMessage(content=prompt, source="user")
result = await agent.on_messages([message], cancellation_token=CancellationToken())
for message in result.inner_messages:
    print(message.content)
display(Markdown(result.chat_message.content))

internet_search useful for when you need to search the internet
copy_file Create a copy of a file in a specified location
file_delete Delete a file
file_search Recursively search for files in a subdirectory that match the regex pattern
move_file Move or rename a file from one location to another
read_file Read file from disk
write_file Write file to disk
list_directory List files and directories in a specified folder
[FunctionCall(id='call_xHjPu8hkT0T3gFoEspX3M1ZC', arguments='{"query":"one-way flight from IAH to BLR June 2025 deals"}', name='internet_search')]
[FunctionExecutionResult(content='Cheap Flights from Houston (IAH) to Bengaluru (BLR) start at $468 for one-way and $795 for round trip. Earn your airline miles on top of our rewards! Find flights to Bengaluru (Bangalore) from $463. Fly from Houston George Bush Airport on Delta, Virgin Atlantic, KLM and more. Find the best deals on flights from Houston (HOUA) to Bengaluru (BLR). Compare prices from hundreds of major travel agent

I have found several promising deals for one-way flights from IAH to BLR in June 2025. Here are the details that will be written to flights.md:

1. **Airline**: Multiple Airlines (Delta, Virgin Atlantic, KLM)
   - **Departure Location**: George Bush Intercontinental Airport (IAH)
   - **Arrival Location**: Kempegowda International Airport (BLR)
   - **Price**: $463
   - **Flight Duration**: 20 hours 30 minutes (with layovers)
   - **Availability**: In June 2025
   - **Booking Options**: Available through various travel agents

2. **Airline**: Various (Air France, etc.)
   - **Departure Location**: Houston (IAH)
   - **Arrival Location**: Bengaluru (BLR)
   - **Price**: $478
   - **Flight Duration**: Not specified
   - **Availability**: In June 2025
   - **Booking Options**: Available through various travel agents

Now I will write these details to flights.md and select the cheapest flight.

In [4]:
# Now we need to call the agent again to write the file

message = TextMessage(content="OK proceed", source="user")

result = await agent.on_messages([message], cancellation_token=CancellationToken())
for message in result.inner_messages:
    print(message.content)
display(Markdown(result.chat_message.content))

[FunctionCall(id='call_SDdNhegP0Sq7DbiaQlr7CNWt', arguments='{"file_path":"flights.md","text":"1. **Airline**: Multiple Airlines (Delta, Virgin Atlantic, KLM)\\n   - **Departure Location**: George Bush Intercontinental Airport (IAH)\\n   - **Arrival Location**: Kempegowda International Airport (BLR)\\n   - **Price**: $463\\n   - **Flight Duration**: 20 hours 30 minutes (with layovers)\\n   - **Availability**: In June 2025\\n   - **Booking Options**: Available through various travel agents\\n\\n2. **Airline**: Various (Air France, etc.)\\n   - **Departure Location**: Houston (IAH)\\n   - **Arrival Location**: Bengaluru (BLR)\\n   - **Price**: $478\\n   - **Flight Duration**: Not specified\\n   - **Availability**: In June 2025\\n   - **Booking Options**: Available through various travel agents","append":false}', name='write_file')]
[FunctionExecutionResult(content='File written successfully to flights.md.', name='write_file', call_id='call_SDdNhegP0Sq7DbiaQlr7CNWt', is_error=False)]


The cheapest one-way flight from IAH to BLR in June 2025 is as follows:

- **Airline**: Multiple Airlines (Delta, Virgin Atlantic, KLM)
- **Departure Location**: George Bush Intercontinental Airport (IAH)
- **Arrival Location**: Kempegowda International Airport (BLR)
- **Price**: $463
- **Flight Duration**: 20 hours 30 minutes (with layovers)
- **Availability**: In June 2025
- **Booking Options**: Available through various travel agents

TERMINATE

In [5]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import  TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat

from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool

serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")


prompt = """Find a one-way non-stop flight from IAH to BLR in June 2025."""


primary_agent = AssistantAgent(
    "primary",
    model_client=model_client,
    tools=[autogen_serper],
    system_message="You are a helpful AI research assistant who looks for promising deals on flights. Incorporate any feedback you receive.",
)

evaluation_agent = AssistantAgent(
    "evaluator",
    model_client=model_client,
    system_message="Provide constructive feedback. Respond with 'APPROVE' when your feedback is addressed.",
)

text_termination = TextMentionTermination("APPROVE")

# With thanks to Peter A for adding in the max_turns - otherwise this can get into a loop..

team = RoundRobinGroupChat([primary_agent, evaluation_agent], termination_condition=text_termination, max_turns=20)


In [6]:
result = await team.run(task=prompt)
for message in result.messages:
    print(f"{message.source}:\n{message.content}\n\n")

user:
Find a one-way non-stop flight from IAH to BLR in June 2025.


primary:
[FunctionCall(id='call_J719ZGyEAqst6GJwWMAqo84B', arguments='{"query":"one-way non-stop flight from IAH (Houston George Bush Intercontinental Airport) to BLR (Kempegowda International Airport) in June 2025"}', name='internet_search')]


primary:
[FunctionExecutionResult(content='Find flights to Bengaluru (Bangalore) from $463. Fly from Houston George Bush Airport on Delta, Virgin Atlantic, KLM and more. Search for Bengaluru ... Missing: non- (Kempegowda. Bengaluru.$546 per passenger.Departing Mon, Jun 23.One-way flight with IndiGo.Outbound indirect flight with IndiGo, departing from Houston George Bush Intercntl ... Missing: non- June. Cheap Flights from Houston (IAH) to Bengaluru (BLR) start at $468 for one-way and $795 for round trip. Earn your airline miles on top of our rewards! Missing: June | Show results with:June. Flights from George Bush Intercontinental Airport to Kempegowda Intl. Airport. IAH to BL

In [10]:
import asyncio
async def setup_fetcher_with_retry(retries=3, delay = 2):
    for attempt in range(retries):
        try:
            fetch_mcp_server = StdioServerParams(command="uvx", args=["mcp-server-fetch"])
            return await mcp_server_tools(fetch_mcp_server)
        except Exception as e:
            if attempt < retries - 1:
                print("Retrying after error {e}")
                await asyncio.sleep(delay)
            else:
                raise                

In [13]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools

# Get the fetch tool from mcp-server-fetch.
# fetch_mcp_server = StdioServerParams(command="uvx", args=["mcp-server-fetch"])
fetcher = await setup_fetcher_with_retry()

# Create an agent that can use the fetch tool.
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
agent = AssistantAgent(name="fetcher", model_client=model_client, tools=fetcher, reflect_on_tool_use=True)  # type: ignore

# Let the agent fetch the content of a URL and summarize it.
result = await agent.run(task="Summarize top 5 news from cnn.com. Reply in Markdown.")
display(Markdown(result.messages[-1].content))

I am currently unable to access CNN's website to retrieve the latest news. However, you can visit [cnn.com](https://www.cnn.com) for the most recent headlines and updates. If there's anything else you would like to know or if you have specific topics in mind, feel free to ask! 

TERMINATE